In [1]:

import pandas as pd

# Load the dataset
df = pd.read_csv("movies.csv", encoding="latin1")

# Display the first 5 rows
print(df.head())

# Display the number of rows and columns
print(df.shape)

# Display all column names
print(df.columns)

# Display information about the dataset
print(df.info())


                                 Name    Year Duration            Genre  \
0                                         NaN      NaN            Drama   
1  #Gadhvi (He thought he was Gandhi)  (2019)  109 min            Drama   
2                         #Homecoming  (2021)   90 min   Drama, Musical   
3                             #Yaaram  (2019)  110 min  Comedy, Romance   
4                   ...And Once Again  (2010)  105 min            Drama   

   Rating Votes            Director       Actor 1             Actor 2  \
0     NaN   NaN       J.S. Randhawa      Manmauji              Birbal   
1     7.0     8       Gaurav Bakshi  Rasika Dugal      Vivek Ghamande   
2     NaN   NaN  Soumyajit Majumdar  Sayani Gupta   Plabita Borthakur   
3     4.4    35          Ovais Khan       Prateik          Ishita Raj   
4     NaN   NaN        Amol Palekar  Rajat Kapoor  Rituparna Sengupta   

           Actor 3  
0  Rajendra Bhatia  
1    Arvind Jangid  
2       Roy Angana  
3  Siddhant Kapoor  
4    

In [2]:

# Remove rows where Rating is missing
df = df.dropna(subset=["Rating"])

# Remove duplicate rows
df = df.drop_duplicates()

# Check the dataset again
print(df.shape)

# Check missing values in each column
print(df.isnull().sum())


(7919, 10)
Name           0
Year           0
Duration    2068
Genre        102
Rating         0
Votes          0
Director       5
Actor 1      125
Actor 2      200
Actor 3      292
dtype: int64


In [3]:

# Find the average rating
print("Average Rating:", df["Rating"].mean())

# Find the highest rating
print("Highest Rating:", df["Rating"].max())

# Find the lowest rating
print("Lowest Rating:", df["Rating"].min())

# Display basic statistics
print(df["Rating"].describe())


Average Rating: 5.841621416845562
Highest Rating: 10.0
Lowest Rating: 1.1
count    7919.000000
mean        5.841621
std         1.381777
min         1.100000
25%         4.900000
50%         6.000000
75%         6.800000
max        10.000000
Name: Rating, dtype: float64


In [4]:

# Select the columns we need
df = df[["Year", "Duration", "Votes", "Genre", "Rating"]].copy()

# Convert Year into a number
df["Year"] = df["Year"].str.replace("(", "", regex=False)
df["Year"] = df["Year"].str.replace(")", "", regex=False)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# Remove 'min' from Duration
df["Duration"] = df["Duration"].str.replace(" min", "", regex=False)
df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")

# Convert Votes into numbers
df["Votes"] = df["Votes"].str.replace(",", "", regex=False)
df["Votes"] = pd.to_numeric(df["Votes"], errors="coerce")

# Fill missing numerical values with their column averages
df["Year"] = df["Year"].fillna(df["Year"].mean())
df["Duration"] = df["Duration"].fillna(df["Duration"].mean())
df["Votes"] = df["Votes"].fillna(df["Votes"].mean())

# Fill missing genres with the most common genre
df["Genre"] = df["Genre"].fillna(df["Genre"].mode()[0])

# Check the result
print(df.head())
print(df.isnull().sum())
print(df.dtypes)


   Year  Duration  Votes                      Genre  Rating
1  2019     109.0      8                      Drama     7.0
3  2019     110.0     35            Comedy, Romance     4.4
5  1997     147.0    827     Comedy, Drama, Musical     4.7
6  2005     142.0   1086        Drama, Romance, War     7.4
8  2012      82.0    326  Horror, Mystery, Thriller     5.6
Year        0
Duration    0
Votes       0
Genre       0
Rating      0
dtype: int64
Year          int64
Duration    float64
Votes         int64
Genre           str
Rating      float64
dtype: object


In [5]:

# Convert Genre into separate columns
df = pd.get_dummies(df, columns=["Genre"], dtype=int)

# Display the first 5 rows
print(df.head())

# Display the new shape
print(df.shape)


   Year  Duration  Votes  Rating  Genre_Action  Genre_Action, Adventure  \
1  2019     109.0      8     7.0             0                        0   
3  2019     110.0     35     4.4             0                        0   
5  1997     147.0    827     4.7             0                        0   
6  2005     142.0   1086     7.4             0                        0   
8  2012      82.0    326     5.6             0                        0   

   Genre_Action, Adventure, Biography  Genre_Action, Adventure, Comedy  \
1                                   0                                0   
3                                   0                                0   
5                                   0                                0   
6                                   0                                0   
8                                   0                                0   

   Genre_Action, Adventure, Crime  Genre_Action, Adventure, Drama  ...  \
1                             

In [6]:

from sklearn.model_selection import train_test_split

# Separate the features and target
X = df.drop("Rating", axis=1)
y = df["Rating"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Display the sizes
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training ratings:", y_train.shape)
print("Testing ratings:", y_test.shape)


Training features: (6335, 435)
Testing features: (1584, 435)
Training ratings: (6335,)
Testing ratings: (1584,)


In [7]:

from sklearn.linear_model import LinearRegression

# Create the model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Predict movie ratings
y_pred = model.predict(X_test)

# Display the first 10 predicted ratings
print("Predicted ratings:", y_pred[:10])

# Display the first 10 actual ratings
print("Actual ratings:", y_test.iloc[:10].values)


Predicted ratings: [5.56454513 6.64098147 5.4218609  6.10141505 4.3752845  5.53982711
 6.17467007 5.78959383 6.39167291 5.78949609]
Actual ratings: [3.3 5.3 5.7 7.2 3.5 7.2 3.8 6.9 5.2 7.4]


In [8]:

from sklearn.metrics import mean_absolute_error, r2_score

# Calculate the prediction error
mae = mean_absolute_error(y_test, y_pred)

# Calculate the R2 score
r2 = r2_score(y_test, y_pred)

# Display the results
print("Mean Absolute Error:", mae)
print("R2 Score:", r2)


Mean Absolute Error: 1.0060170415687288
R2 Score: 0.14367261101680961


In [9]:

# Create a table of actual and predicted ratings
results = pd.DataFrame({
    "Actual Rating": y_test.values,
    "Predicted Rating": y_pred
})

# Display the first 10 results
print(results.head(10).round(2))


   Actual Rating  Predicted Rating
0            3.3              5.56
1            5.3              6.64
2            5.7              5.42
3            7.2              6.10
4            3.5              4.38
5            7.2              5.54
6            3.8              6.17
7            6.9              5.79
8            5.2              6.39
9            7.4              5.79
